In [30]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [31]:
import pandas as pd
from tqdm import tqdm

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [32]:
# lets generate the index for search
from ingest import build_index, load_faq_data

documents = load_faq_data()
documents_llm_zoomcamp = [] 
for d in documents:
    if d["course"] == "llm-zoomcamp":
        documents_llm_zoomcamp.append(d)

documents = documents_llm_zoomcamp
index = build_index(documents)

# search function using text
def text_search(query):
    boost_dict = {
        "question": 3.0,
        "section": 0.5
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [33]:
# function that returns a relevante list for each query
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [34]:
relevance_total = compute_relevance_total(ground_truth, text_search)

100%|██████████| 805/805 [00:02<00:00, 283.61it/s]


In [35]:
relevance_total[:5]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [36]:
# HIT RATE

example = relevance_total[:5]

cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

2

In [37]:
print(cnt / len(example))

0.4


In [38]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [39]:
#hit_rate(relevance_total) # 0.8347826086956521
hit_rate(example) # 0.9333333333333333

0.4

In [40]:
# MRR Mean Reciprocal Rank

# Logic for MRR
l = [0,0,1,0,0] # 33%
l = [0,0,0,1,0] # 25%
l = [0,0,0,0,1] # 20%
l = [1,0,0,0,0] # 100%
l = [0,1,0,0,0] # 50%
l=example[1]

print(l)

weight = 0.0
total_count = 0.0
for x in range(len(l)):
    if l[x] == 1:
        weight = 1 / (x + 1)
        break
print(weight)

[0, 0, 0, 0, 0]
0.0


In [41]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score
total_score / len(example)

0.4

In [42]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [43]:
mrr(example)
# 0.822

0.4

In [44]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [45]:
evaluate(
    ground_truth,
    text_search
)

  1%|          | 7/805 [00:00<00:13, 57.48it/s]

100%|██████████| 805/805 [00:02<00:00, 275.76it/s]


{'hit_rate': 0.8347826086956521, 'mrr': 0.7110144927536229}

In [46]:
def search_boost(query, question_boost):
    boost_dict = {
        "question": question_boost,
        "section": 0.5
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [48]:
for boost in [0.5,1.0,3.0,5.0,10.0]:
    result = evaluate(
    ground_truth,
    #search_boost
    lambda query, boost=boost: search_boost(query, boost)
    
)
    print(f"boost={boost}: {result}")

  0%|          | 0/805 [00:00<?, ?it/s]

100%|██████████| 805/805 [00:02<00:00, 294.02it/s]


boost=0.5: {'hit_rate': 0.8894409937888199, 'mrr': 0.7915942028985508}


100%|██████████| 805/805 [00:02<00:00, 293.02it/s]


boost=1.0: {'hit_rate': 0.9006211180124224, 'mrr': 0.7926501035196688}


100%|██████████| 805/805 [00:02<00:00, 279.34it/s]


boost=3.0: {'hit_rate': 0.8347826086956521, 'mrr': 0.7110144927536229}


100%|██████████| 805/805 [00:02<00:00, 294.31it/s]


boost=5.0: {'hit_rate': 0.7950310559006211, 'mrr': 0.6821946169772254}


100%|██████████| 805/805 [00:02<00:00, 294.72it/s]

boost=10.0: {'hit_rate': 0.7627329192546584, 'mrr': 0.6525672877846785}


In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )